<h1 style="text-align: center;">Perceptrón multicapa - Clasificación de tejidos</h1>
<h3 style="text-align: center;">Dr. Carlos Francisco Méndez Cruz</h3>

Notebook adaptado de la [Capacitación oficial de PyTorch.](https://pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html)

------------------------------------------------------------------------


# Objetivo de la práctica

Entrenar un modelo de Perceptrón Multicapa (_MultiLayer Perceptron_, MLP) para predecir el tejido de una muestra biológica usando su perfil de expresión.

# Descripción de la tarea
El modelo debe clasificar perfiles de expresión en 7 posibles tejidos:

- Cerebelo
- Colon
- Endometrio
- Hipocampo
- Riñón
- Hígado
- Placenta

# Conjunto de datos
Usaremos el conjunto de datos llamado [tissue_gene_expression](https://rdrr.io/cran/dslabs/man/tissue_gene_expression.html).

------------------------------------------------------------------------


# Importación de librerías

In [48]:
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

True

------------------------------------------------------------------------

# Manejo de datos

## 1. De CSV a DataFrame

Usaremos el archivo tissue_gene_expression.csv disponible en la plataforma del curso.



In [49]:
df_data = pd.read_csv('/mnt/c/Users/vrahu/Documents/Ismadlsh/CCG/2026-S5/Data_Science/data/tissue_gene_expression.csv')
print("Dimensionalidad de datos:", df_data.shape)
print("Categorías verdaderas únicas:", df_data.Tissue.unique())
df_data.head()


Dimensionalidad de datos: (189, 501)
Categorías verdaderas únicas: <StringArray>
['Cerebelo', 'Colon', 'Endometrio', 'Hipocampo', 'Riñón', 'Hígado',
 'Placenta']
Length: 7, dtype: str


,MAML1,LHPP,SEPT10,B3GNT4,ZNF280D,SOX12,C21orf62,PER3,HOXA10,HOXC5,...,EPHA1,MFGE8,OAZ2,PCBP3,POLA1,KREMEN2,CYP7B1,LILRB3,GSAP,Tissue
0,9.825680,8.327163,5.499382,8.692371,5.642305,6.254751,5.840938,8.330565,5.520803,7.660495,...,7.017747,8.439859,10.233591,8.457219,7.347544,8.391655,4.760017,8.230182,6.740385,Cerebelo
1,9.631247,8.542827,5.644292,8.833679,5.685083,6.289460,6.065646,8.263576,5.528077,7.569041,...,7.098755,8.627736,9.840514,8.784941,7.476239,8.260828,4.952373,8.332502,6.390683,Cerebelo
2,9.690548,8.476486,5.717187,8.498623,5.958232,6.215924,5.771117,9.102294,5.599405,7.418460,...,6.742484,8.191681,10.060973,8.523428,7.299006,8.275763,4.796232,7.978194,6.677038,Cerebelo
3,9.991592,8.506430,5.789430,8.421143,5.736563,6.512055,5.851617,9.210058,5.561197,7.332254,...,6.850879,8.280976,9.913842,8.210854,7.453407,8.276021,4.963021,7.894200,6.815246,Cerebelo
4,9.578130,8.371783,5.779698,8.818455,5.630886,6.201556,5.965732,8.502172,5.601099,7.535668,...,6.838669,8.043418,10.028252,8.758295,7.253716,8.473418,4.871638,8.129660,6.996641,Cerebelo


## 2. Separación de datos en entrenamiento y prueba

In [50]:
df_train, df_test = train_test_split(df_data, test_size=0.2, random_state=42, stratify=df_data.Tissue)
print("Dimensionalidad de datos de entrenamiento:", df_train.shape)
print("Dimensionalidad de datos de prueba:", df_test.shape)


Dimensionalidad de datos de entrenamiento: (151, 501)
Dimensionalidad de datos de prueba: (38, 501)


## 3. De Pandas DataFrame a PyTorch DataSet

### Diccionarios para convertir categorías en ids y viceversa

In [4]:
categorias_en_ids = {
    "Cerebelo": 0,
    "Colon": 1,
    "Endometrio": 2,
    "Hipocampo": 3,
    "Riñón": 4,
    "Hígado": 5,
    "Placenta": 6,
}

ids_en_categorias = {
    0: "Cerebelo",
    1: "Colon",
    2: "Endometrio",
    3: "Hipocampo",
    4: "Riñón",
    5: "Hígado",
    6: "Placenta",
}


In [5]:
print(categorias_en_ids['Cerebelo'])

0


In [6]:
print(ids_en_categorias[1])

Colon


### Creación de clase DataSet

In [51]:
# Definición de la clase a partir del módulo Dataset de PyTorch
class PerfilesDataset(Dataset):
    def __init__(self, dataframe):
        # Definimos dos atributos de la clase:
        # - El DataFrame de donde crearemos el DataSet
        self.data = dataframe
        # - El número de registros del DataSet
        self.len = len(dataframe)

    def __getitem__(self, index):
        # El argumento "index" es un índice numérico desde 0 hasta el último registro el DataFrame

        # Guardamos en una variable el perfil de expresión separado de la categoría
        # El procedimiento "drop" con "axis = 1" elimina una columna del DataFrame
        # El procedimiento "iloc" localiza y entrega el registro "index" del DataFrame
        perfil_de_expresion = self.data.drop(['Tissue'], axis=1).iloc[index].values
        # Guardamos en una variable la categoría separada del perfil de expresión
        categoria = self.data.Tissue.iloc[index]

        # Convertimos la categoría en id con nuestro diccionario
        encoded_categoria = categorias_en_ids[categoria]

        # Convertimos variables con el perfil y la categoría en tensores de PyTorch
        tensor_perfil_de_expresion = torch.as_tensor(perfil_de_expresion)
        tensor_categoria = torch.as_tensor(encoded_categoria)
        return tensor_perfil_de_expresion, tensor_categoria

    def __len__(self):
        # Procedimiento obligatorio para regresar el número de renglones del DataSet
        return self.len


### Creación de objetos dataset

In [52]:
dataset_train = PerfilesDataset(dataframe=df_train)
dataset_test = PerfilesDataset(dataframe=df_test)

## 4. De PyTorch DataSet a PyTorch DataLoader

Para definir el DataLoader, necesitamos establecer un tamaño de lote (`batch_size`). La red neuronal ajustará sus pesos cada lote.

In [53]:
batch_size = 8

dataloader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
dataloader_test = DataLoader(dataset_test, batch_size=batch_size, shuffle=False)


### Observar dimensionalidad y contenido del DataLoader


In [54]:
for X, y in dataloader_train:
    print("Dimensionalidad de datos del primer lote del DataLoader de entrenamiento:", X.shape)
    print("Dimensionalidad de categorías del primer lote del DataLoader de entrenamiento:", y.shape)
    print("Ids de categorías del primer lote del DataLoader de entrenamiento:", y)
    break

Dimensionalidad de datos del primer lote del DataLoader de entrenamiento: torch.Size([8, 500])
Dimensionalidad de categorías del primer lote del DataLoader de entrenamiento: torch.Size([8])
Ids de categorías del primer lote del DataLoader de entrenamiento: tensor([5, 4, 1, 5, 6, 3, 4, 6])


------------------------------------------------------------------------


# Definición de `device` para entrenamiento

In [55]:
if torch.cuda.is_available():
    device="cuda"
else: 
    device="cpu"
print("Device disponible para entrenamiento:", device)

Device disponible para entrenamiento: cuda


# Diseño de la arquitectura de la red neuronal


Creamos nuestra clase `MLP` a partir de la clase base de PyTorch [nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html). Nuestra clase será definida de acuerdo a la arquitectura de red que deseamos entrenar.

Debemos definir:
- Capas
- Nodos por capa
- Función de activación
- Procesamiento hacia adelante de la red (`forward`)


In [56]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(500, 67),
            nn.ReLU(),
            nn.Linear(67, 68),
            nn.ReLU(),
            nn.Linear(68, 69),
            nn.ReLU(),
            nn.Linear(69, 7)
        )
        self.double()
    def forward(self, x):
        logits = self.capas(x)
        return logits

## Creación de red neuronal en `device`
Usando la clase definida anteriormente, creamos nuestra red neuronal e indicamos que la red será entrenada en el `device` definido anteriormente.

In [57]:
red_mlp = MLP().to(device)
print(red_mlp)

MLP(
  (capas): Sequential(
    (0): Linear(in_features=500, out_features=67, bias=True)
    (1): ReLU()
    (2): Linear(in_features=67, out_features=68, bias=True)
    (3): ReLU()
    (4): Linear(in_features=68, out_features=69, bias=True)
    (5): ReLU()
    (6): Linear(in_features=69, out_features=7, bias=True)
  )
)


## Definición de función de pérdida, optimizador y tasa de aprendizaje

La [función de pérdida (loss function)](https://pytorch.org/docs/stable/nn.html#loss-functions) más recomendada para resolver tareas de clasificación multiclase es la Entropía cruzada (`CrossEntropyLoss`).

El [optimizador](https://pytorch.org/docs/stable/optim.html) más sencillo es uno basado de el algortimo de Descenso por gradiente estocástico (`SGD`).

La tasa de aprendizaje (learning rate o `lr`) será definida como 0.001 (`1e-3`).

In [58]:
funcion_de_perdida = nn.CrossEntropyLoss()
optimizador_de_pesos = torch.optim.Adam(red_mlp.parameters(), lr=0.0001)

# Entrenamiento y evaluación de la red neuronal

## Definición de función de entrenamiento (`train`)

Esto incluye procesar los registros del `DataLoader` lote por lote "hacia delante" y calcular el valor de la función de pérdida. Luego ejecutaremos el paso "hacia atrás" o backpropagation para calcular los gradientes. Finalmente ejecutaremos el ajuste de los pesos de la red. En cada iteración, la red deberá predecir cada vez mejor.


In [59]:
def train(dataloader, red, funcion_perdida, optimizador):
    total_de_registros = len(dataloader.dataset)
    # Activamos nuestra red en modo de entrenamiento
    red.train()

    # Para cada batch de datos y categorías:
    for lote, (X, y) in enumerate(dataloader):
        # Enviamos datos y categorías al device
        X, y = X.to(device), y.to(device)
        # Obtenemos predicciones de la red (logits) para el conjunto de datos del lote (forward)
        logits = red(X)
        # Obtenemos valor de función de pérdida
        valor_de_perdida = funcion_perdida(logits, y)
        # Backpropagation
        valor_de_perdida.backward()
        # Modificación (ajuste) de pesos
        optimizador.step()
        # Reseteo de gradientes para evitar acumulación
        optimizador.zero_grad()

        # Cada 5 batches
        if lote % 5 == 0:
            print("Lote número:", lote)
            # Calculamos registros procesados multiplicando el número de lote (`lote`) por la cantidad de registros del lote (`X`)
            registros_procesados = (lote + 1) * len(X)
            # item() regresa el valor del tensor en tipo numérico de Python
            print("Valor de función de pérdida:", valor_de_perdida.item(), "Lote:", lote, "Registros:[", registros_procesados, "/", total_de_registros, "]")


## Definición de función de prueba (`test`)

Esto incluye procesar los datos de prueba lote por lote con la red previamente entrenada. Para cada lote se calculará el valor de la función de perdida y el procentaje de aciertos de las predicciones de la red (`Accuracy`).


In [62]:
def test(dataloader, red, funcion_perdida):
    total_de_registros = len(dataloader.dataset)
    # Obtenemos el número de lotes de los datos de prueba
    numero_de_lotes = len(dataloader)
    # Activamos nuestra red en modo de prueba
    red.eval()

    # Inicializamos valor de función de pérdida y número de predicciones correctas
    valor_de_perdida_acumulado = 0
    predicciones_correctas_acumuladas = 0

    # Desactivamos los cálculos de gradientes porque no estamos en fase de entrenamiento
    with torch.no_grad():
        # Para cada conjunto de datos y categorías del lote:
        for X, y in dataloader:
            # Enviamos datos y categorías al device
            X, y = X.to(device), y.to(device)
            # Obtenemos predicciones de la red (logits) para el conjunto de datos del lote
            logits = red(X)
            # Obtenemos el valor de la función de pérdida acumulado y predicciones correctas acumuladas
            valor_de_perdida = funcion_de_perdida(logits, y).item()
            valor_de_perdida_acumulado = valor_de_perdida_acumulado + valor_de_perdida
            # Obtenemos predicciones correctas (True|False) comparando el índice del valor máximo de los logits con la categoría verdadera
            # argmax(1) regresa el índice del valor máximo en el tensor.
            predicciones_correctas = (logits.argmax(1) == y)
            # Convertimos predicciones correctas en números
            predicciones_correctas = predicciones_correctas.type(torch.float)
            # Sumamos predicciones correctas
            predicciones_correctas = predicciones_correctas.sum().item()
            # Obtenemos predicciones correctas acumuladas
            predicciones_correctas_acumuladas = predicciones_correctas_acumuladas + predicciones_correctas
    #test_loss /= num_batche
    #correct /= size
    accuracy  = predicciones_correctas_acumuladas / total_de_registros
    print("Accuracy:", accuracy)
    #print(f"Fase de prueba: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    return accuracy

------------------------------------------------------------------------


## Entrenamiento de la red neuronal

Ejecutamos las funciones de entrenamiento (`train`) y prueba (`test`) por un número de épocas (veces que la red entrena usando todos los datos disponibles).

La función `train` imprimirá el valor de pérdida por cada 5 lotes. La función `test` imprimirá el valor de pérdida y la exactitud de predicción.


In [ ]:
epocas = 30
for t in range(epocas):
    print("\nEpoca", t+1)
    print("Entrenamiento\n-------------------------------")
    train(dataloader_train, red_mlp, funcion_de_perdida, optimizador_de_pesos)
    print("\nPrueba\n-------------------------------")
    test(dataloader_test, red_mlp, funcion_de_perdida)
print("Done!")


Epoca 1
Entrenamiento
-------------------------------
Lote número: 0
Valor de función de pérdida: 1.8499907089566194 Lote: 0 Registros:[ 8 / 151 ]
Lote número: 5
Valor de función de pérdida: 1.915288392005756 Lote: 5 Registros:[ 48 / 151 ]
Lote número: 10
Valor de función de pérdida: 1.8970252561100527 Lote: 10 Registros:[ 88 / 151 ]
Lote número: 15
Valor de función de pérdida: 1.8763258595958106 Lote: 15 Registros:[ 128 / 151 ]

Prueba
-------------------------------
Accuracy: 0.23684210526315788

Epoca 2
Entrenamiento
-------------------------------
Lote número: 0
Valor de función de pérdida: 1.9142396992018744 Lote: 0 Registros:[ 8 / 151 ]
Lote número: 5
Valor de función de pérdida: 1.880614772521406 Lote: 5 Registros:[ 48 / 151 ]
Lote número: 10
Valor de función de pérdida: 1.9597919334885086 Lote: 10 Registros:[ 88 / 151 ]
Lote número: 15
Valor de función de pérdida: 1.7563482420454992 Lote: 15 Registros:[ 128 / 151 ]

Prueba
-------------------------------
Accuracy: 0.3684210526

In [67]:
err=0.0000001
conver=0
accuracy_b =0 
while conver<3:
    print("\nEntrenamiento\n-------------------------------")
    train(dataloader_train, red_mlp, funcion_de_perdida, optimizador_de_pesos)
    print("\nPrueba\n-------------------------------")
    accuracy=test(dataloader_test, red_mlp, funcion_de_perdida)
    if abs(accuracy-accuracy_b)<err:
        conver+=1
    accuracy_b=accuracy


Entrenamiento
-------------------------------
Lote número: 0
Valor de función de pérdida: 0.680116563410964 Lote: 0 Registros:[ 8 / 151 ]
Lote número: 5
Valor de función de pérdida: 0.5829869208745432 Lote: 5 Registros:[ 48 / 151 ]
Lote número: 10
Valor de función de pérdida: 0.6291298537895988 Lote: 10 Registros:[ 88 / 151 ]
Lote número: 15
Valor de función de pérdida: 0.6562776498418947 Lote: 15 Registros:[ 128 / 151 ]

Prueba
-------------------------------
Accuracy: 0.8157894736842105

Entrenamiento
-------------------------------
Lote número: 0
Valor de función de pérdida: 0.6760742178595074 Lote: 0 Registros:[ 8 / 151 ]
Lote número: 5
Valor de función de pérdida: 0.52783610119453 Lote: 5 Registros:[ 48 / 151 ]
Lote número: 10
Valor de función de pérdida: 0.5838150065696022 Lote: 10 Registros:[ 88 / 151 ]
Lote número: 15
Valor de función de pérdida: 1.2423967873038693 Lote: 15 Registros:[ 128 / 151 ]

Prueba
-------------------------------
Accuracy: 0.7894736842105263

Entrenamie

## Guardar modelo entrenado

Para guardar nuestro modelo entrenado, almacenamos en un archivo los pesos ajustados finales de la red neuronal.


In [68]:
torch.save(red_mlp.state_dict(), "/mnt/c/Users/vrahu/Documents/Ismadlsh/CCG/2026-S5/Data_Science/src/models/model_NN67_predic_tissue.mod")
print("Mejor modelo guardado!")

Mejor modelo guardado!


------------------------------------------------------------------------


# Uso del modelo para inferencia


## Cargar mejor modelo entrenado desde archivo

Primero, creamos una instancia de la red neuronal y luego le pasamos los pesos almacenados.


In [69]:
mejor_modelo = MLP().to(device)
mejor_modelo.load_state_dict(torch.load("/mnt/c/Users/vrahu/Documents/Ismadlsh/CCG/2026-S5/Data_Science/src/models/model_NN67_predic_tissue.mod"))

<positron-console-cell-69>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


<All keys matched successfully>

## Usar mejor modelo entrenado



Definimos función de predicción. Usamos nuestro mejor modelo entrenado en modo de evaluación `model.eval()` y sin ajustes de pesos: `torch.no_grad()`.

Imprimimos la predicción del modelo y la categoría verdadera.

In [70]:
def predict(dataloader, modelo):
    # Activamos nuestra red en modo de prueba
    modelo.eval()
    # Desactivamos los cálculos de gradientes porque no estamos en fase de entrenamiento
    with torch.no_grad():
        for X, y in dataloader:
            # Enviamos datos y categorías al device
            X, y = X.to(device), y.to(device)
            # Obtenemos predicciones de la red (logits) para los datos del lote
            logits = modelo(X)
            # Imprimimos los logits
            print("\nLogits:", logits)
            # Obtenemos índice correspondiente al valor máximo de los logits
            indice_logit_maximo = logits[0].argmax(0)
            # Obtenemos categoría predicha con el índice del valor máximo de los logits
            categoria_predicha = ids_en_categorias[indice_logit_maximo.item()]
            print("Índice logit máximo:", indice_logit_maximo.item(), "que corresponde a categoría predicha:", categoria_predicha)
            categoria_verdadera = y
            # Obtenemos categoría verdadera
            tejido = ids_en_categorias[categoria_verdadera.item()]
            print("Id de categoría verdadera:", categoria_verdadera, "que corresponde a la categoría:", tejido)
            print("-------------------------------")


In [71]:
dataset_predict = PerfilesDataset(dataframe=df_test.iloc[5:10])
dataloader_predict = DataLoader(dataset_predict, batch_size=1, shuffle=False)

In [72]:
predict(dataloader_predict, mejor_modelo)


Logits: tensor([[-3.5843,  1.0565, -0.7599, -5.7927, -0.7247, -1.7839, -2.0435]],
       device='cuda:0', dtype=torch.float64)
Índice logit máximo: 1 que corresponde a categoría predicha: Colon
Id de categoría verdadera: tensor([1], device='cuda:0') que corresponde a la categoría: Colon
-------------------------------

Logits: tensor([[  2.9753,  -3.8731,  -0.5284,   3.6608,  -1.8172, -11.4374,  -1.3647]],
       device='cuda:0', dtype=torch.float64)
Índice logit máximo: 3 que corresponde a categoría predicha: Hipocampo
Id de categoría verdadera: tensor([3], device='cuda:0') que corresponde a la categoría: Hipocampo
-------------------------------

Logits: tensor([[  4.3726,  -3.1869,  -0.5818,   2.6973,  -2.6478, -12.3240,  -1.7053]],
       device='cuda:0', dtype=torch.float64)
Índice logit máximo: 0 que corresponde a categoría predicha: Cerebelo
Id de categoría verdadera: tensor([0], device='cuda:0') que corresponde a la categoría: Cerebelo
-------------------------------

Logits: 

------------------------------------------------------------------------
